<a href="https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/yolo_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Traffic Sign Detection & Road Safety Monitor
### Cloud Training, Metric Verification & ONNX Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shubhamjena28/traffic-sign-recognition-road-safety/blob/main/train/yolo_colab.ipynb)

This notebook provides a complete training pipeline for **YOLOv8 Traffic Sign Object Detection**.

### Target Classes
- `0: stop` (Stop signs)
- `1: speed` (Speed limit signs: 20, 30, 50, 60, 70, 80, 100, 120 km/h)
- `2: danger` (Warning signs: curves, road works, pedestrian crossing, signals, etc.)

### Evaluation Targets
| Metric | Target Requirement |
|---|---|
| **mAP@0.5** | $\ge 0.80$ |
| **Missed-sign rate** (stop, speed, danger) | $\le 5\%$ |
| **CPU inference latency** | $\le 150$ ms/frame |
| **Deployed model size** | $\le 50$ MB |

## Step 1: Check GPU Acceleration
Make sure your Colab runtime is set to GPU (*Runtime -> Change runtime type -> T4 GPU*).

In [9]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Thu Oct 29 06:09:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             29W /   70W |     455MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Clone Repository & Install Dependencies

In [10]:
import os
if not os.path.exists("traffic-sign-recognition-road-safety"):
    !git clone https://github.com/shubhamjena28/traffic-sign-recognition-road-safety.git

%cd traffic-sign-recognition-road-safety

# Install Ultralytics YOLO and ONNX export dependencies
!pip install --quiet ultralytics onnx onnxruntime onnxslim opencv-python matplotlib pandas pyyaml

Cloning into 'traffic-sign-recognition-road-safety'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 206 (delta 51), reused 185 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 169.10 KiB | 1.32 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/traffic-sign-recognition-road-safety/traffic-sign-recognition-road-safety


## Step 3: Configure YOLO Dataset
Generates a Colab-compatible dataset configuration pointing to local paths.

In [11]:
from pathlib import Path
import yaml

base_data = Path("/content/traffic-sign-recognition-road-safety/data")
if not base_data.exists():
    base_data = Path.cwd() / "data"

# Check if dataset images exist, if not, create starter annotated set
train_imgs = list((base_data / "images" / "train").glob("*.png")) + list((base_data / "images" / "train").glob("*.jpg"))
if len(train_imgs) == 0:
    print("[INFO] Generating starter annotated traffic sign dataset...")
    !python data/create_sample_dataset.py

# Write Colab dataset configuration
config = {
    "path": str(base_data.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "stop",
        1: "speed",
        2: "danger"
    }
}

colab_yaml_path = base_data / "traffic_signs_colab.yaml"
with open(colab_yaml_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"[SUCCESS] Colab configuration written to: {colab_yaml_path}")
with open(colab_yaml_path) as f:
    print(f.read())

[SUCCESS] Colab configuration written to: /content/traffic-sign-recognition-road-safety/data/traffic_signs_colab.yaml
names:
  0: stop
  1: speed
  2: danger
path: /content/traffic-sign-recognition-road-safety/data
train: images/train
val: images/val



*(Optional)* **Download Full GTSDB Detection Dataset**
Uncomment below to download and convert the official German Traffic Sign Detection Benchmark archive (~1.6 GB) for full training.

In [12]:
# !python data/download_datasets.py --dataset gtsdb
# !python data/convert_gtsdb_to_yolo.py --gtsdb-dir data/raw/gtsdb --output-dir data

## Step 4: Train YOLOv8 Detector
We start from the official pretrained `yolov8n.pt` (nano) weights. It satisfies the strict `< 50 MB` size target and `< 150 ms` latency requirement.

In [13]:
from ultralytics import YOLO

# Initialize model
model = YOLO("yolov8n.pt")

# Run training
results = model.train(
    data=str(colab_yaml_path),
    epochs=30,
    imgsz=640,
    batch=16,
    patience=10,
    project="runs/detect",
    name="colab_yolo_model",
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/traffic-sign-recognition-road-safety/data/traffic_signs_colab.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=colab_yolo_mode

## Step 5: Evaluate Against All Target Metrics

In [14]:
import time
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# YOLO saved the weights under a nested path in the training step
weights_path = Path("runs/detect/runs/detect/colab_yolo_model/weights/best.pt")
val_model = YOLO(str(weights_path))

# 1. Run validation
metrics = val_model.val(data=str(colab_yaml_path), imgsz=640, device="cpu")
map50 = float(metrics.box.map50)
recall = float(metrics.box.mr)
missed_rate = (1.0 - recall) * 100.0

# 2. Check deployed model size on disk
size_mb = weights_path.stat().st_size / (1024 * 1024)

# 3. Benchmark CPU Latency (50 frames)
test_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
for _ in range(5):
    _ = val_model.predict(test_img, device="cpu", verbose=False)

times = []
for _ in range(30):
    t0 = time.perf_counter()
    _ = val_model.predict(test_img, device="cpu", verbose=False)
    times.append((time.perf_counter() - t0) * 1000.0)

cpu_latency = float(np.mean(times))

print("\n" + "="*65)
print("             YOLO TRAFFIC SIGN BENCHMARK RESULTS")
print("="*65)
print(f"Metric                  Target          Achieved        Status")
print("-"*65)
status_map = "PASS" if map50 >= 0.80 else "TRAIN_MORE_EPOCHS"
status_miss = "PASS" if missed_rate <= 5.0 else "TRAIN_MORE_EPOCHS"
status_lat = "PASS" if cpu_latency <= 150.0 else "FAIL"
status_sz = "PASS" if size_mb <= 50.0 else "FAIL"

print(f"mAP@0.5                 >= 0.80         {map50:.4f}          {status_map}")
print(f"Missed-sign rate        <= 5.0%         {missed_rate:.2f}%          {status_miss}")
print(f"CPU Latency             <= 150 ms       {cpu_latency:.2f} ms        {status_lat}")
print(f"Model Size              <= 50 MB        {size_mb:.2f} MB         {status_sz}")
print("="*65)

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2475.7±740.2 MB/s, size: 63.4 KB)
val: Scanning /content/traffic-sign-recognition-road-safety/data/labels/val.cache... 9 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9/9 2.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.5s/it 2.5s
                   all          9          9      0.848      0.922      0.995      0.966
                  stop          3          3      0.911          1      0.995      0.995
                 speed          3          3      0.632          1      0.995      0.995
                danger          3          3          1      0.766      0.995      0.907
Speed: 5.7ms preprocess, 166.4ms inference, 0.0ms loss, 97.8ms postprocess per image
Results saved to /content/

## Step 6: Export Model to ONNX
Export the model to ONNX format for deployment with Java and OpenCV DNN.

In [15]:
onnx_export_file = val_model.export(format="onnx", imgsz=640, opset=14, simplify=True)
onnx_file_path = Path(onnx_export_file)
onnx_size_mb = onnx_file_path.stat().st_size / (1024 * 1024)
print(f"[SUCCESS] Exported ONNX: {onnx_file_path} ({onnx_size_mb:.2f} MB)")

Ultralytics 8.4.192 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,133,193 parameters, 0 gradients, 8.6 GFLOPs

PyTorch: starting from 'runs/detect/runs/detect/colab_yolo_model/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 14...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 1.1s, saved as 'runs/detect/runs/detect/colab_yolo_model/weights/best.onnx' (12.2 MB)

Export complete (1.3s)
Results saved to /content/traffic-sign-recognition-road-safety/traffic-sign-recognition-road-safety/runs/detect/runs/detect/colab_yolo_model/weights/best.onnx
Predict:         yolo predict task=detect model=runs/detect/runs/detect/colab_yolo_model/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=runs/detect/runs/detect/colab_yolo_model/weights/best.onnx imgsz=640 data=/content/traffic-sign-recognition-road-safety/data

## Step 7: Visualize Results & Predictions

In [16]:
from PIL import Image
import matplotlib.pyplot as plt

run_dir = Path("runs/detect/colab_yolo_model")
for fname in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    img_path = run_dir / fname
    if img_path.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(Image.open(img_path))
        plt.title(fname)
        plt.axis("off")
        plt.show()

## Step 8: Download Trained Weights
Downloads `best.pt` and `best.onnx` to your computer for immediate use in your Java and Python application.

In [17]:
from google.colab import files

print("Downloading best.pt ...")
files.download(str(weights_path))

if onnx_file_path.exists():
    print("Downloading best.onnx ...")
    files.download(str(onnx_file_path))

print("[DONE] Both weight files downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[DONE] Both weight files downloaded.
